# Transformer Encoder Block: Multi-Head Attention + Residuals + LayerNorm + MLP

This notebook builds one **Transformer encoder block** (Vaswani et al.,
2017, "Attention Is All You Need", Sec. 3.1, 3.2.2, 3.3) on top of the
single-head self-attention exercise. One encoder layer has two
sub-layers, each wrapped the same way (the paper's literal post-LN
wrapping, Sec. 3.1):

$$\text{sub-layer output} = \text{LayerNorm}(x + \text{Sublayer}(x))$$

Sub-layer 1, multi-head self-attention ($Q=K=V=X$):

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1,\dots,\text{head}_h)W^O,\quad
\text{head}_i = \text{softmax}\left(\frac{Q_iK_i^\top}{\sqrt{d_k}}\right)V_i,\quad d_k=d_v=D/h$$

Sub-layer 2, position-wise feed-forward network (Sec. 3.3):

$$\text{FFN}(y) = \max(0, yW_1+b_1)W_2+b_2$$

Full block ($X$: input, $Z$: output, both `[B,N,D]` -- the block is
shape-preserving):

$$\text{attn\_out} = \text{MultiHead}(X,X,X) \quad\rightarrow\quad Y = \text{LayerNorm}(X+\text{attn\_out}) \quad\rightarrow\quad \text{ffn\_out} = \text{FFN}(Y) \quad\rightarrow\quad Z = \text{LayerNorm}(Y+\text{ffn\_out})$$

| symbol | code variable | shape | meaning |
|---|---|---|---|
| $X$ | `x` | `[B,N,D]` | block input tokens |
| $h$ | `num_heads` | scalar | number of attention heads |
| $d_k=d_v$ | `head_dim` | scalar | `D // h` |
| $Q,K,V$ (per head) | `q,k,v` | `[B,h,N,d_k]` | split-head projections |
| attn weights | `attention_weights` | `[B,h,N,N]` | per-head softmax rows |
| $\text{Concat}(\text{heads})$ | `context` | `[B,N,D]` | merged heads before $W^O$ |
| $W^O$ | `W_o` | `[D,D]` | output mixing projection |
| sub-layer 1 out | `y` | `[B,N,D]` | $\text{LayerNorm}(X+\text{attn\_out})$ |
| sub-layer 2 out | `z` | `[B,N,D]` | $\text{LayerNorm}(y+\text{ffn\_out})$, block output |

**No dropout**: the paper's Sec. 5.4 training-time regularization is
omitted because this is an untrained forward-mechanics demo, not
training -- it does not affect the equations being reproduced.

Core code lives in `code/transformers/transformer_block.py`
(`MultiHeadSelfAttention`, `PositionwiseFeedForward`,
`TransformerEncoderBlock`, `print_block_shapes`), which reuses
`manual_softmax` and `make_toy_token_sequence` from
`self_attention_qkv.py` rather than duplicating them. Unit-checked with
`python code/transformers/transformer_block.py --smoke-test`. This
notebook is a thin orchestration layer.

## 1. Clone/pull repository

Transformer block code lives in `code/transformers/transformer_block.py` (and its dependency `self_attention_qkv.py`), not duplicated in this notebook. Replace `REPO_URL` with this repository's URL.

In [ ]:
import os

REPO_URL = "https://github.com/RoeiZ/from_zero_to_hero.git"
REPO_DIR = "/content/from_zero_to_hero"

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "code", "transformers"))

## 2. Build a toy token sequence (no dataset)

Reuses the same six-token toy "sentence" (`cat`/`dog`, `car`/`truck`, `apple`/`banana`) from the self-attention exercise, so results are directly comparable to the single-head case.

In [ ]:
import torch

from transformer_block import (
    TransformerEncoderBlock,
    print_block_shapes,
)
from self_attention_qkv import make_toy_token_sequence, visualize_attention_matrix

x, tokens = make_toy_token_sequence(seed=0)  # x: [1,N,8]
print(f"x.shape = {tuple(x.shape)}")
print(f"tokens = {tokens}")

## 3. Run the Transformer encoder block and print `[B,N,D]` at every step

`TransformerEncoderBlock` runs multi-head self-attention, a residual + LayerNorm, the position-wise FFN, and a second residual + LayerNorm. Every intermediate stays `[B,N,D]` -- the block is shape-preserving.

In [ ]:
torch.manual_seed(0)
NUM_HEADS = 2
FFN_HIDDEN_DIM = 16

block = TransformerEncoderBlock(embed_dim=x.shape[-1], num_heads=NUM_HEADS, ffn_hidden_dim=FFN_HIDDEN_DIM)
intermediates = block(x)

print_block_shapes(x, intermediates)

## 4. Visualize one head's attention matrix

Multi-head attention weights are `[B,h,N,N]`; indexing `[0,0]` selects batch element 0, head 0, giving the same `[N,N]` shape the single-head visualization expects. See the self-attention notebook's interpretation caveat -- these projections are untrained, so the pattern reflects random initialization, not learned semantics.

In [ ]:
HEAD_TO_PLOT = 0
attention_weights_head0 = intermediates["attn_attention_weights"][0, HEAD_TO_PLOT]  # [B,h,N,N] -> [N,N]

visualize_attention_matrix(attention_weights_head0, tokens)